# 05c — Embedded Methods

## The One-Sentence Version
Some models do feature selection as a side effect of training.

## What You'll Build Intuition For
- How Lasso (L1) drives feature weights to exactly zero
- How tree-based models rank features by split importance
- The difference between L1 (hard selection) and L2 (soft shrinkage)
- When to select features vs transform them (PCA)

## Prerequisites
- [05b — Wrapper Methods](05b_wrapper_methods.ipynb) — testing feature combinations
- [03c — PCA Applied](../03_linear_methods/03c_pca_applied.ipynb) — for the selection vs transformation comparison

## The Story

Filter methods score features one at a time. Wrapper methods test combinations but are expensive. Embedded methods are the Goldilocks option: the model itself decides which features matter *during training*.

The most elegant example is Lasso regression. It adds a penalty to the training objective that naturally drives unimportant feature weights to exactly zero. You don't need a separate feature selection step — the model tells you which features it needs by keeping their weights nonzero.

Tree-based models like Random Forests do something similar: features that are never useful for splitting get zero importance. The model's structure reveals which features matter.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Lasso, Ridge, ElasticNet, LassoCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

from utils.plotting import apply_style, COLOURS, PALETTE
from utils.data_generators import make_patient_data

apply_style()
rng = np.random.default_rng(42)

In [ ]:
# Dataset: 5 signal + 20 noise features, continuous target
X, feature_names, signal_mask = make_patient_data(
    n=500, d_signal=5, d_noise=20, seed=42
)

# Continuous target: combination of signal features + noise
y = 0.8 * X[:, 0] + 0.5 * X[:, 1] - 0.3 * X[:, 2] + rng.normal(0, 5, 500)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Features: {len(feature_names)} ({signal_mask.sum()} signal, {(~signal_mask).sum()} noise)")
print(f"Target is a linear combination of first 3 signal features + noise.")

## Lasso Regression (L1 Regularisation)

Normal linear regression assigns a weight to every feature. Lasso adds a penalty proportional to the *absolute value* of the weights. This penalty has a remarkable property: it drives irrelevant weights to exactly zero. The features that survive with nonzero weights are the important ones.

In [ ]:
# Coefficient paths: watch weights shrink to zero as regularisation increases
alphas = np.logspace(-3, 1, 50)
coef_paths = []

for alpha in alphas:
    lasso = Lasso(alpha=alpha, max_iter=10000)
    lasso.fit(X_scaled, y)
    coef_paths.append(lasso.coef_.copy())

coef_paths = np.array(coef_paths)

fig, ax = plt.subplots(figsize=(12, 6))
for j in range(X_scaled.shape[1]):
    colour = COLOURS["signal"] if signal_mask[j] else COLOURS["noise"]
    lw = 2.0 if signal_mask[j] else 0.7
    alpha_line = 0.9 if signal_mask[j] else 0.3
    label = feature_names[j] if signal_mask[j] else None
    ax.plot(alphas, coef_paths[:, j], color=colour, linewidth=lw,
            alpha=alpha_line, label=label)

ax.set_xscale("log")
ax.axhline(y=0, color=COLOURS["neutral"], linestyle="-", linewidth=0.5)
ax.set_xlabel("Regularisation strength (α)")
ax.set_ylabel("Coefficient value")
ax.set_title("Lasso coefficient paths: noise features (red) hit zero first")
ax.legend(loc="upper right", fontsize=9)
plt.tight_layout()
plt.savefig("visuals/05c_lasso_paths.png", dpi=150, bbox_inches="tight")
plt.show()

print("As regularisation increases (left → right):")
print("  - Noise feature weights (thin red) hit zero quickly")
print("  - Signal feature weights (thick blue) survive longer")
print("  - The last features standing are the truly important ones")

In [ ]:
# Use cross-validation to pick the best alpha
lasso_cv = LassoCV(cv=5, random_state=42, max_iter=10000)
lasso_cv.fit(X_scaled, y)

selected = np.abs(lasso_cv.coef_) > 1e-6
print(f"Best α: {lasso_cv.alpha_:.4f}")
print(f"Features selected: {selected.sum()} / {len(feature_names)}")
print(f"\nSelected features and their coefficients:")
for name, coef, is_sig in zip(feature_names, lasso_cv.coef_, signal_mask):
    if abs(coef) > 1e-6:
        tag = "SIGNAL" if is_sig else "noise"
        print(f"  {name}: {coef:+.3f} ({tag})")

## Tree-Based Importance

Random Forests offer a different kind of embedded selection. Each tree splits the data on the most useful features. Features that are used more often, and that produce bigger improvements when they split, get higher importance scores.

In [ ]:
# Binary target for classification
y_cls = (X[:, 0] > np.median(X[:, 0])).astype(int)

rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_scaled, y_cls)

importances = rf.feature_importances_
order = np.argsort(importances)[::-1]

fig, ax = plt.subplots(figsize=(12, 5))
bar_colours = [COLOURS["signal"] if signal_mask[i] else COLOURS["noise"] for i in order]
ax.barh([feature_names[i] for i in order], importances[order],
        color=bar_colours, alpha=0.8, edgecolor="white")
ax.set_xlabel("Feature Importance (Gini)")
ax.set_title("Random Forest: signal features dominate importance")
ax.invert_yaxis()

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(facecolor=COLOURS["signal"], label="Signal"),
    Patch(facecolor=COLOURS["noise"], label="Noise"),
], loc="lower right")

plt.tight_layout()
plt.savefig("visuals/05c_rf_importance.png", dpi=150, bbox_inches="tight")
plt.show()

print("Signal features cluster at the top — the forest correctly identifies them.")
print("\nCaveat: tree importance is biased toward high-cardinality and continuous features.")
print("Use permutation importance for a more reliable estimate.")

## L1 vs L2 vs Elastic Net

Three flavours of regularisation:
- **L1 (Lasso):** drives weights to zero → hard feature selection
- **L2 (Ridge):** shrinks weights toward zero but never reaches it → soft shrinkage
- **Elastic Net:** a mix of both

In [ ]:
# Compare L1, L2, Elastic Net coefficients
alpha_val = 0.5

lasso = Lasso(alpha=alpha_val, max_iter=10000)
ridge = Ridge(alpha=alpha_val)
enet = ElasticNet(alpha=alpha_val, l1_ratio=0.5, max_iter=10000)

lasso.fit(X_scaled, y)
ridge.fit(X_scaled, y)
enet.fit(X_scaled, y)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, model, name in zip(axes, [lasso, ridge, enet],
                            ["Lasso (L1)", "Ridge (L2)", "Elastic Net"]):
    coefs = model.coef_
    bar_colours = [COLOURS["signal"] if signal_mask[j] else COLOURS["noise"]
                   for j in range(len(coefs))]
    ax.bar(range(len(coefs)), np.abs(coefs), color=bar_colours, alpha=0.8,
           edgecolor="white")
    ax.set_title(f"{name}\n({(np.abs(coefs) > 1e-6).sum()} nonzero)")
    ax.set_xlabel("Feature index")
    ax.set_ylabel("|Coefficient|")
    # Mark signal features on x-axis
    ax.set_xticks(range(len(coefs)))
    ax.set_xticklabels(["S" if s else "" for s in signal_mask], fontsize=7)

fig.suptitle("L1 zeros out noise; L2 shrinks but keeps everything; Elastic Net is in between",
             fontweight="bold")
plt.tight_layout()
plt.savefig("visuals/05c_l1_vs_l2.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Lasso (L1): {(np.abs(lasso.coef_) > 1e-6).sum()} nonzero coefficients")
print(f"Ridge (L2): {(np.abs(ridge.coef_) > 1e-6).sum()} nonzero coefficients")
print(f"Elastic Net: {(np.abs(enet.coef_) > 1e-6).sum()} nonzero coefficients")
print(f"\nL1 is the only one that produces exact zeros — true feature selection.")

## Feature Selection vs Transformation

We now have two complete toolkits:
- **Selection** (this module): keep original features, discard the rest
- **Transformation** (PCA, etc.): create new features from combinations of originals

When should you use which?

In [ ]:
# Visual: selection keeps interpretable features, PCA creates opaque ones
from sklearn.decomposition import PCA
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression

y_cls = (X[:, 0] > np.median(X[:, 0])).astype(int)

# Selection: top 5 by Lasso
lasso_sel = Lasso(alpha=0.3, max_iter=10000)
lasso_sel.fit(X_scaled, y)
top5_idx = np.argsort(np.abs(lasso_sel.coef_))[::-1][:5]
X_selected = X_scaled[:, top5_idx]
selected_names = [feature_names[i] for i in top5_idx]

# Transformation: PCA to 5 components
pca = PCA(n_components=5)
X_pca = pca.fit_transform(X_scaled)

# Compare accuracy
model = LogisticRegression(max_iter=1000, random_state=42)
acc_selected = cross_val_score(model, X_selected, y_cls, cv=5).mean()
acc_pca = cross_val_score(model, X_pca, y_cls, cv=5).mean()
acc_all = cross_val_score(model, X_scaled, y_cls, cv=5).mean()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy comparison
labels = ["All 25\nfeatures", "Top 5\n(Lasso)", "PCA\n(5 components)"]
accs = [acc_all, acc_selected, acc_pca]
bar_colours = [COLOURS["neutral"], COLOURS["signal"], COLOURS["accent"]]
ax1.bar(labels, accs, color=bar_colours, alpha=0.8, edgecolor="white")
for i, a in enumerate(accs):
    ax1.text(i, a + 0.01, f"{a:.1%}", ha="center", fontsize=12, fontweight="bold")
ax1.set_ylabel("CV Accuracy")
ax1.set_ylim(0, 1.1)
ax1.set_title("Accuracy: selection ≈ transformation")

# Interpretability comparison
interp_labels = ["Selected\nfeatures", "PCA\ncomponents"]
interp_examples = [
    "\n".join(selected_names[:3]),
    "PC1 = 0.3×Age + 0.2×HR + ...\nPC2 = -0.1×Age + 0.4×SpO2 + ..."
]
ax2.text(0.25, 0.7, "Lasso selects:", fontsize=13, fontweight="bold",
         transform=ax2.transAxes, ha="center")
ax2.text(0.25, 0.4, "\n".join(selected_names), fontsize=11,
         transform=ax2.transAxes, ha="center", color=COLOURS["signal"],
         family="monospace")
ax2.text(0.75, 0.7, "PCA creates:", fontsize=13, fontweight="bold",
         transform=ax2.transAxes, ha="center")
ax2.text(0.75, 0.4, "PC1 = 0.3×Age + 0.2×HR + ...\nPC2 = -0.1×Age + ...\nPC3 = ???",
         fontsize=11, transform=ax2.transAxes, ha="center", color=COLOURS["accent"],
         family="monospace")
ax2.set_xlim(0, 1)
ax2.set_ylim(0, 1)
ax2.set_xticks([])
ax2.set_yticks([])
ax2.set_title("Interpretability: selection wins")

fig.suptitle("Selection vs Transformation: similar accuracy, different trade-offs",
             fontweight="bold")
plt.tight_layout()
plt.savefig("visuals/05c_selection_vs_transformation.png", dpi=150, bbox_inches="tight")
plt.show()

print("Selection: 'Age and Heart Rate matter most' → actionable")
print("PCA: 'PC1 is elevated' → opaque")
print(f"\nBoth use 5 features/components and achieve similar accuracy.")
print(f"Choose based on whether interpretability or raw performance matters more.")

| Need | Method |
|------|--------|
| Interpretability ("which features matter?") | **Feature selection** (Lasso, RF importance) |
| Maximum performance, opaqueness OK | **Transformation** (PCA, autoencoders) |
| Clinical/regulatory requirements | **Feature selection** (must name specific variables) |
| Huge feature space (>10k) | **Filters** first, then embedded |
| Small feature space (<50) | **Wrappers** or embedded |

<details>
<summary><b>The Maths Behind This</b></summary>

**Lasso (L1) objective:** $\min_w \frac{1}{2n}\|\mathbf{y} - X\mathbf{w}\|^2 + \alpha \sum_j |w_j|$

The L1 penalty $\sum_j |w_j|$ has a geometric explanation for why it produces zeros: the constraint region $\{\mathbf{w} : \sum |w_j| \leq t\}$ is a diamond (in 2D) or cross-polytope (in higher D). The optimal point where the elliptical loss contours first touch this diamond is most likely at a corner — where one or more coordinates are exactly zero.

**Ridge (L2) objective:** $\min_w \frac{1}{2n}\|\mathbf{y} - X\mathbf{w}\|^2 + \alpha \sum_j w_j^2$

The L2 constraint region is a sphere. The elliptical contours touch a sphere at a smooth point, not a corner — so coefficients shrink but never reach zero.

**Elastic Net:** $\min_w \frac{1}{2n}\|\mathbf{y} - X\mathbf{w}\|^2 + \alpha [\rho \sum_j |w_j| + \frac{1-\rho}{2} \sum_j w_j^2]$

The $\rho$ parameter (`l1_ratio`) controls the blend. Higher $\rho$ → more sparsity.

</details>

## Where This Matters

**Sepsis prediction:** A clinical team asks "which 5 lab values best predict sepsis?" Lasso regression is the ideal tool — it directly outputs a sparse set of features with interpretable coefficients. The doctor gets: "lactate > 2, WBC > 12k, creatinine > 1.5 are the strongest predictors."

**Genomics:** With 20,000 genes, Elastic Net is standard for gene expression studies. Pure Lasso is too aggressive (picks one gene from a correlated group and ignores the rest). Elastic Net keeps correlated genes together while still zeroing out the irrelevant thousands.

## Feynman Check

1. **Why does L1 regularisation zero out features while L2 only shrinks them?**

2. **When should you use feature selection instead of PCA?**

3. **A clinical team asks which 5 lab values best predict sepsis. Which method do you use and why?**

---

Module 05 is complete. You now know three approaches to feature selection: filters (fast, individual), wrappers (thorough, expensive), and embedded (built into training).

In **Module 06: Learned Compression**, we'll move beyond linear methods entirely — using neural networks to learn nonlinear compressions of data.